# MINNEMUDAC 2026: Complete Data-Cleaning Pipeline

This notebook preserves the original Excel workbooks, creates temporary working copies,
cleans each dataset with explicit rules, and saves reproducible CSV intermediates.
Ambiguous records are flagged instead of silently deleted or imputed.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import tempfile

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

SOURCE_DIR = Path.cwd()
if not (SOURCE_DIR / "Organization.xlsx").exists():
    candidate = SOURCE_DIR / "MINNEMUDAC_2026" / "Data"
    if candidate.exists():
        SOURCE_DIR = candidate

OUTPUT_DIR = SOURCE_DIR / "cleaned"
CSV_DIR = OUTPUT_DIR / "_csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "dictionary": SOURCE_DIR / "Data_Dictionary.xlsx",
    "organization": SOURCE_DIR / "Organization.xlsx",
    "food_shelf_stats": SOURCE_DIR / "org_FSStats.xlsx",
    "market_bucks": SOURCE_DIR / "marketbucksUsage.xlsx",
    "incoming_calls": SOURCE_DIR / "Incoming Call.xlsx",
}

missing_files = [str(path) for path in FILES.values() if not path.exists()]
assert not missing_files, f"Missing source files: {missing_files}"

inventory = pd.DataFrame({
    "dataset": FILES.keys(),
    "file": [path.name for path in FILES.values()],
    "size_mb": [round(path.stat().st_size / 1_048_576, 2) for path in FILES.values()],
})
display(inventory)


## 1. Create unlocked working copies

OneDrive can lock the original workbooks. PowerShell copies them to a temporary folder,
allowing pandas to read them without changing the originals.


In [ ]:
WORK_DIR = Path(tempfile.gettempdir()) / "minnemudac_working"
WORK_DIR.mkdir(parents=True, exist_ok=True)
WORK_FILES = {}

for name, source_path in FILES.items():
    destination = WORK_DIR / source_path.name
    command = (
        f"Copy-Item -LiteralPath '{source_path}' "
        f"-Destination '{destination}' -Force"
    )
    subprocess.run(
        ["powershell.exe", "-NoProfile", "-Command", command],
        check=True,
    )
    WORK_FILES[name] = destination

print(f"Working copies: {WORK_DIR}")


## 2. Reusable cleaning functions


In [ ]:
def strip_outer_whitespace_and_blanks(df, dataset_name):
    """Trim outer whitespace and convert blank-only strings to missing values."""
    result = df.copy(deep=True)
    records = []
    for column in result.columns:
        text_mask = result[column].map(lambda value: isinstance(value, str))
        if not text_mask.any():
            continue
        original = result.loc[text_mask, column]
        stripped = original.str.strip()
        blank_mask = stripped.eq("")
        trim_mask = stripped.ne("") & original.ne(stripped)
        if blank_mask.any():
            result.loc[blank_mask.index[blank_mask], column] = pd.NA
        if trim_mask.any():
            result.loc[trim_mask.index[trim_mask], column] = stripped.loc[trim_mask]
        if blank_mask.any() or trim_mask.any():
            records.append({
                "dataset": dataset_name,
                "column": column,
                "rule": "Trim outer whitespace; blank strings become missing",
                "values_changed": int(blank_mask.sum() + trim_mask.sum()),
            })
    return result, records


def parse_datetime_naive(series):
    """Parse datetimes and store UTC timestamps without timezone metadata for Excel."""
    parsed = pd.to_datetime(series, errors="coerce", utc=True)
    return parsed.dt.tz_convert(None)


def normalize_zip(value):
    """Return five-digit or ZIP+4 text without treating ZIP as a measurement."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)
    digits = re.sub(r"\D", "", text)
    if len(digits) <= 5 and digits:
        return digits.zfill(5)
    if len(digits) == 9:
        return f"{digits[:5]}-{digits[5:]}"
    return text


def normalize_identifier(value):
    """Preserve identifiers as text, including meaningful decimal suffixes such as 13462.1."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    return re.sub(r"\.0$", "", text)


def convert_numeric_when_safe(df, exclusions=()):
    """Convert a column only when every populated value is numeric-like."""
    result = df.copy()
    exclusions = set(exclusions)
    for column in result.columns:
        if column in exclusions:
            continue
        populated = result[column].notna()
        if not populated.any():
            continue
        converted = pd.to_numeric(result[column], errors="coerce")
        if converted.loc[populated].notna().all():
            nonmissing = converted.dropna()
            if len(nonmissing) and np.allclose(nonmissing % 1, 0):
                result[column] = converted.astype("Int64")
            else:
                result[column] = converted.astype("Float64")
    return result


def normalize_contact_methods(value):
    """Normalize free-text contact-method combinations into stable pipe-separated values."""
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    text = re.sub(r"\b(or|and)\b", ",", text)
    parts = [part.strip() for part in text.split(",") if part.strip()]
    normalized = []
    for part in parts:
        if part in {"phone", "call", "phone call"}:
            label = "Phone"
        elif part == "text":
            label = "Text"
        elif part == "email":
            label = "Email"
        else:
            label = part.title()
        if label not in normalized:
            normalized.append(label)
    order = ["Phone", "Text", "Email"]
    normalized = [x for x in order if x in normalized] + [x for x in normalized if x not in order]
    return "|".join(normalized) if normalized else pd.NA


def dataframe_schema(df):
    schema = {}
    for column in df.columns:
        dtype = str(df[column].dtype)
        if dtype.startswith("datetime"):
            kind = "date"
        elif dtype in {"Int64", "int64", "int32"}:
            kind = "integer"
        elif dtype in {"Float64", "float64", "float32"}:
            kind = "number"
        elif dtype in {"bool", "boolean"}:
            kind = "boolean"
        else:
            kind = "text"
        schema[str(column)] = kind
    return schema


## 3. Load all raw datasets


In [ ]:
org_raw = pd.read_excel(WORK_FILES["organization"], sheet_name="Organization", dtype=object)
fs_raw = pd.read_excel(WORK_FILES["food_shelf_stats"], sheet_name="org_FSStats", dtype=object)
mb_raw = pd.read_excel(WORK_FILES["market_bucks"], sheet_name="marketbucksUsage", dtype=object)
calls_raw = pd.read_excel(WORK_FILES["incoming_calls"], sheet_name="Incoming Call", dtype=object)

raw_summary = pd.DataFrame([
    {"dataset": "Organization", "rows": len(org_raw), "columns": org_raw.shape[1]},
    {"dataset": "org_FSStats", "rows": len(fs_raw), "columns": fs_raw.shape[1]},
    {"dataset": "marketbucksUsage", "rows": len(mb_raw), "columns": mb_raw.shape[1]},
    {"dataset": "Incoming Call", "rows": len(calls_raw), "columns": calls_raw.shape[1]},
])
display(raw_summary)


## 4. Clean Organization

Organization is cleaned first because it supplies identifiers used by the food-shelf and
Market Bucks datasets. Three ZIP values found in `State` are moved to `Zip`; their state is
left missing and flagged rather than guessed.


In [ ]:
cleaning_records = []
org, records = strip_outer_whitespace_and_blanks(org_raw, "Organization")
cleaning_records.extend(records)

org["dq_zip_moved_from_state"] = False
state_text = org["State"].astype("string")
zip_in_state = state_text.str.fullmatch(r"\d{5}", na=False) & org["Zip"].isna()
org.loc[zip_in_state, "Zip"] = state_text.loc[zip_in_state]
org.loc[zip_in_state, "State"] = pd.NA
org.loc[zip_in_state, "dq_zip_moved_from_state"] = True

org["Zip"] = org["Zip"].map(normalize_zip)
org["MailZip"] = org["MailZip"].map(normalize_zip)
org["block_geoid"] = org["block_geoid"].map(
    lambda value: pd.NA if pd.isna(value) else re.sub(r"\.0$", "", str(value)).zfill(15)
)

org_identifier_columns = [
    "__ID", "_id_type", "_id_subtype", "_id_Bank", "_id_Agency",
    "_id_IfFoodBank", "_id_findhelpMapCat",
]
for column in org_identifier_columns:
    org[column] = org[column].map(normalize_identifier)

org["dq_invalid_supershelf_status"] = org["SuperShelfStatus"].eq("1")
org.loc[org["dq_invalid_supershelf_status"], "SuperShelfStatus"] = pd.NA
org["dq_invalid_district_section"] = org["DistrictSection"].eq("8")
org.loc[org["dq_invalid_district_section"], "DistrictSection"] = pd.NA
org["dq_missing_agency_type"] = org["AgencyType"].isna() | org["_id_type"].isna()

org_date_columns = [
    "zzCreationTS", "zzModificationTS", "InactiveDate", "TaxEINVerifiedDate",
    "TefapActiveDate", "TefapInActiveDate", "SuperShelfCertDate",
]
for column in org_date_columns:
    org[column] = parse_datetime_naive(org[column])

org = convert_numeric_when_safe(
    org,
    exclusions={
        "Zip", "MailZip", "block_geoid", "State", "MailState",
        *org_date_columns, *org_identifier_columns,
    },
)
for column in [
    "dq_zip_moved_from_state", "dq_invalid_supershelf_status",
    "dq_invalid_district_section", "dq_missing_agency_type",
]:
    org[column] = org[column].astype("boolean")
assert org["__ID"].notna().all() and org["__ID"].is_unique

display(org[["__ID", "City", "State", "Zip", "dq_zip_moved_from_state"]].head())


## 5. Clean monthly food-shelf statistics

Duplicate site-month groups, date disagreements, negative service measures, and unmatched
organization keys are made explicit. Duplicate grain records are retained for review.


In [ ]:
fs, records = strip_outer_whitespace_and_blanks(fs_raw, "org_FSStats")
cleaning_records.extend(records)

for column in ["DateForSearch", "zzCreationTS", "zzModificationTS"]:
    fs[column] = parse_datetime_naive(fs[column])

fs_identifier_columns = ["__ID", "_id_site", "_id_bank", "id_agency"]
for column in fs_identifier_columns:
    fs[column] = fs[column].map(normalize_identifier)

fs = convert_numeric_when_safe(
    fs,
    exclusions={
        "DateForSearch", "zzCreationTS", "zzModificationTS", "County", "FoodBank", "AgencyType",
        *fs_identifier_columns,
    },
)

grain = ["_id_site", "_Year", "_Month"]
fs["dq_duplicate_site_month"] = fs.duplicated(grain, keep=False)
fs["dq_date_mismatch"] = (
    fs["DateForSearch"].dt.year.ne(pd.to_numeric(fs["_Year"], errors="coerce"))
    | fs["DateForSearch"].dt.month.ne(pd.to_numeric(fs["_Month"], errors="coerce"))
)

protected_numeric = {"__ID", "_id_site", "_Year", "_Month", "_YearMonth", "_id_bank", "id_agency"}
measure_columns = [
    column for column in fs.columns
    if column not in protected_numeric and pd.api.types.is_numeric_dtype(fs[column])
]
negative_masks = {column: fs[column].lt(0).fillna(False) for column in measure_columns}
fs["dq_negative_value_corrected"] = False
fs["dq_negative_columns"] = pd.NA
for row_index in fs.index:
    affected = [column for column, mask in negative_masks.items() if bool(mask.loc[row_index])]
    if affected:
        fs.loc[row_index, "dq_negative_value_corrected"] = True
        fs.loc[row_index, "dq_negative_columns"] = "|".join(affected)
        fs.loc[row_index, affected] = pd.NA

organization_ids = set(org["__ID"].dropna().astype(str))
site_ids = fs["_id_site"].astype("string")
agency_ids = fs["id_agency"].astype("string")
fs["dq_unmatched_site_id"] = site_ids.notna() & ~site_ids.isin(organization_ids)
fs["dq_unmatched_agency_id"] = agency_ids.notna() & ~agency_ids.isin(organization_ids)

assert fs["__ID"].notna().all() and fs["__ID"].is_unique


## 6. Clean Market Bucks usage

Multiple rows for the same site and date are flagged rather than aggregated because the
source context permits multiple submissions and redemption of benefits from earlier visits.


In [ ]:
mb, records = strip_outer_whitespace_and_blanks(mb_raw, "marketbucksUsage")
cleaning_records.extend(records)

for column in ["zzCreationTS", "zzCreationDate", "zzModificationTS", "Dated"]:
    mb[column] = parse_datetime_naive(mb[column])

for column in ["_ID", "_id_site"]:
    mb[column] = mb[column].map(normalize_identifier)

mb = convert_numeric_when_safe(
    mb,
    exclusions={"zzCreationTS", "zzCreationDate", "zzModificationTS", "Dated", "_ID", "_id_site"},
)
mb["dq_duplicate_site_date"] = mb.duplicated(["_id_site", "Dated"], keep=False)
mb["dq_event_after_creation"] = mb["Dated"].dt.date > mb["zzCreationTS"].dt.date
mb_site = mb["_id_site"].astype("string")
mb["dq_unmatched_site_id"] = mb_site.notna() & ~mb_site.isin(organization_ids)

assert mb["_ID"].notna().all() and mb["_ID"].is_unique


## 7. Clean Incoming Calls

Duplicate IDs are resolved by keeping the most recently modified, most complete record.
Known category variants are standardized. Ambiguous values and workflow inconsistencies
remain visible through data-quality flags.


In [ ]:
calls, records = strip_outer_whitespace_and_blanks(calls_raw, "Incoming Call")
cleaning_records.extend(records)

call_date_columns = [
    "zzCreationTS", "zzModificationTS", "zzModificationTSNotServer", "zzLastArchived",
    "Date", "TimeStamp", "Follow up Date", "FollowupDate2",
]
for column in call_date_columns:
    calls[column] = parse_datetime_naive(calls[column])

calls["__ID"] = calls["__ID"].map(normalize_identifier)

duplicate_ids = set(calls.loc[calls["__ID"].duplicated(keep=False), "__ID"])
calls["source_duplicate_id"] = calls["__ID"].isin(duplicate_ids)
calls["_completeness"] = calls.notna().sum(axis=1)
calls["_source_order"] = np.arange(len(calls))
calls = (
    calls.sort_values(
        ["__ID", "zzModificationTS", "_completeness", "_source_order"],
        na_position="first",
    )
    .drop_duplicates("__ID", keep="last")
    .sort_values("_source_order")
    .drop(columns=["_completeness", "_source_order"])
    .reset_index(drop=True)
)

calls["Call Answered"] = calls["Call Answered"].replace({
    "1st Call": "1st",
    "2nd Call": "2nd",
})
calls["County"] = calls["County"].replace({"Saint Louis": "St. Louis"})
calls["BestContactMethod"] = calls["BestContactMethod"].map(normalize_contact_methods)

calls = convert_numeric_when_safe(calls, exclusions=set(call_date_columns) | {"CallTime", "__ID"})
calls["NumberInHouse"] = pd.to_numeric(calls["NumberInHouse"], errors="coerce").astype("Int64")

calls["dq_date_timestamp_mismatch"] = calls["Date"].dt.date != calls["TimeStamp"].dt.date
calls["dq_followup_before_call"] = calls["Follow up Date"] < calls["Date"]
calls["dq_followup_date_without_needed"] = calls["Follow up Date"].notna() & calls["FollowUpNeeded"].isna()
calls["dq_needed_without_followup_date"] = calls["FollowUpNeeded"].notna() & calls["Follow up Date"].isna()

known_snap_status = {
    "Currently Enrolled", "Not Reached", "Materials Provided", "Screened-Eligible",
    "Applied", "Screened", "Not Applicable", "Screened-Not Eligible", "Declined", "Denied",
}
calls["dq_unrecognized_snap_status"] = (
    calls["SNAP Status"].notna() & ~calls["SNAP Status"].isin(known_snap_status)
)

assert calls["__ID"].notna().all() and calls["__ID"].is_unique


## 8. Validate and save reproducible intermediates


In [ ]:
datasets = {
    "Organization_clean": org,
    "org_FSStats_clean": fs,
    "marketbucksUsage_clean": mb,
    "Incoming_Call_clean": calls,
}

quality_summary = pd.DataFrame([
    {
        "dataset": "Organization",
        "raw_rows": len(org_raw),
        "clean_rows": len(org),
        "primary_key_unique": org["__ID"].is_unique,
        "flagged_rows": int(org.filter(regex=r"^dq_").any(axis=1).sum()),
    },
    {
        "dataset": "org_FSStats",
        "raw_rows": len(fs_raw),
        "clean_rows": len(fs),
        "primary_key_unique": fs["__ID"].is_unique,
        "flagged_rows": int(fs.filter(regex=r"^dq_").any(axis=1).sum()),
    },
    {
        "dataset": "marketbucksUsage",
        "raw_rows": len(mb_raw),
        "clean_rows": len(mb),
        "primary_key_unique": mb["_ID"].is_unique,
        "flagged_rows": int(mb.filter(regex=r"^dq_").any(axis=1).sum()),
    },
    {
        "dataset": "Incoming Call",
        "raw_rows": len(calls_raw),
        "clean_rows": len(calls),
        "primary_key_unique": calls["__ID"].is_unique,
        "flagged_rows": int(calls.filter(regex=r"^dq_").any(axis=1).sum()),
    },
])

for name, df in datasets.items():
    df.to_csv(CSV_DIR / f"{name}.csv", index=False, date_format="%Y-%m-%d %H:%M:%S")

quality_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False)
pd.DataFrame(cleaning_records).to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False)

schemas = {name: dataframe_schema(df) for name, df in datasets.items()}
(CSV_DIR / "schemas.json").write_text(json.dumps(schemas, indent=2), encoding="utf-8")

from openpyxl import Workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

def export_clean_dataframe(df, sheet_name, output_path, schema):
    workbook = Workbook(write_only=True)
    sheet = workbook.create_sheet(sheet_name)
    sheet.freeze_panes = "A2"
    sheet.sheet_view.showGridLines = False
    headers = [str(column) for column in df.columns]

    sample = df.head(300)
    for index, column in enumerate(headers, start=1):
        lengths = sample[column].dropna().astype(str).str.len()
        sample_width = int(lengths.max()) if len(lengths) else 0
        sheet.column_dimensions[get_column_letter(index)].width = min(35, max(10, len(column) + 2, sample_width + 2))

    header_cells = []
    for header in headers:
        cell = WriteOnlyCell(sheet, value=header)
        cell.font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        header_cells.append(cell)
    sheet.append(header_cells)

    for values in df.itertuples(index=False, name=None):
        output_row = []
        for column, value in zip(headers, values):
            kind = schema.get(column, "text")
            if pd.isna(value):
                value = None
            elif isinstance(value, pd.Timestamp):
                value = value.to_pydatetime()
            elif isinstance(value, np.generic):
                value = value.item()
            cell = WriteOnlyCell(sheet, value=value)
            if kind == "integer":
                cell.number_format = "#,##0"
            elif kind == "number":
                cell.number_format = "#,##0.00"
            elif kind == "date":
                cell.number_format = "yyyy-mm-dd hh:mm:ss"
            output_row.append(cell)
        sheet.append(output_row)

    last_column = get_column_letter(len(headers))
    sheet.auto_filter.ref = f"A1:{last_column}{len(df) + 1}"
    workbook.save(output_path)
    print(f"Saved {output_path.name}: {len(df):,} rows, {len(headers)} columns")

excel_outputs = {
    "Organization_clean": ("Organization", OUTPUT_DIR / "Organization_clean.xlsx"),
    "org_FSStats_clean": ("org_FSStats", OUTPUT_DIR / "org_FSStats_clean.xlsx"),
    "marketbucksUsage_clean": ("marketbucksUsage", OUTPUT_DIR / "marketbucksUsage_clean.xlsx"),
    "Incoming_Call_clean": ("Incoming Call", OUTPUT_DIR / "Incoming_Call_clean.xlsx"),
}

for name, df in datasets.items():
    sheet_name, output_path = excel_outputs[name]
    export_clean_dataframe(df, sheet_name, output_path, schemas[name])

display(quality_summary)
print(f"Finished. Cleaned Excel workbooks are in: {OUTPUT_DIR}")
